# Decision Tree

## Modules

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

## Functions

In [5]:
def compute_entropy(y):
    if len(y)==0:
        return 0
    
    classes, counts=np.unique(y, return_counts=True)
    probablities= counts / len(y)
    entropy= -np.sum(probablities * np.log2(probablities))
    return entropy

def split_node(X, node_indices, feature, threshold):
    left_indices=[]
    right_indices=[]

    for i in node_indices:
        if X[i][feature]<=threshold:
            left_indices.append(i)
        else:
            right_indices.append(i)

    return left_indices, right_indices

def compute_infogain(X, Y, node_indices, feature, threshold):
    left_indices, right_indices= split_node(X, node_indices, feature, threshold)
    if len(left_indices) == 0 or len(right_indices) == 0:
            return 0
    node_entropy=compute_entropy(Y[node_indices])
    right_entropy=compute_entropy(Y[right_indices])
    left_entropy=compute_entropy(Y[left_indices])
    r_p=len(right_indices)/len(node_indices)
    l_p=len(left_indices)/len(node_indices)
    infogain= node_entropy - (right_entropy*r_p+left_entropy*l_p)
    return infogain

def get_best_split(X,Y, node_indices):
    max_infogain=0
    t=None
    f=None
    for i in range(X.shape[1]):
        list_values=np.unique(X[node_indices, i])
        for j in list_values:
            infogain=compute_infogain(X,Y, node_indices, i, j)
            if infogain> max_infogain:
                max_infogain=infogain
                t=j
                f=i
    return t,f

def build_tree(X,Y, node_indices, depth=0, max_depth=5):
    if depth==max_depth or len(np.unique(Y[node_indices]))==1:
        labels,counts = np.unique(Y[node_indices], return_counts=True)
        return labels[np.argmax(counts)]
    
    threshold, feature=get_best_split(X,Y,node_indices)
    left_indices, right_indices=split_node(X,node_indices,feature,threshold)
    if len(left_indices) == 0 or len(right_indices) == 0:
        labels,counts = np.unique(Y[node_indices], return_counts=True)
        return labels[np.argmax(counts)]
    
    left_subtree=build_tree(X,Y,left_indices,depth+1,max_depth)
    right_subtree=build_tree(X,Y,right_indices,depth+1,max_depth)
    
    return {'feature': feature, 'threshold': threshold, 'left': left_subtree, 'right': right_subtree}

def predict(X, tree):
    predictions=[]
    for x in X:
        current_node=tree
        while isinstance(current_node, dict):
            feature=current_node['feature']
            threshold=current_node['threshold']
            if x[feature]<=threshold:
                current_node=current_node['left']
            else:
                current_node=current_node['right']
        predictions.append(current_node)
    return np.array(predictions)



## how the functions work:

1. compute_entropy():
* this checks the randomness of a node by using the entropy formula -sigma(p.log2(p)) where p is the proportions of each class in the node. 
* if there is nothing in the node it returns the entropy is 0
* takes the count of each unique class in the node and divides this count with the total length of the node giving an array of proportions of each class

2. split_node():
* basically splits the root node into 2 daughter nodes. 
* all the values in the root node containing value of feature below or equal to a certain threshold value gets thrown into the left indice and other gets thrown in the right indice.

3. compute_infogain():
* basically checks how much information is gained from the split or how less random the system began after splitting.
* for this it computes the entropy of each daughter node and multiply each entropy with its proportion wrt to the root node and is subtracted from the entropy of the root node.
* follows eqn infogain= root_entropy - (wleft* entropy_left + wright* entropy_right) 

4. get_best_split():
* to choose basically the best feature and best threshold to be used for splitting so that the split maximizes the infogain as much as possible.
* for this the model splits the root node for every feature and for all possible values in that feature as threshold calculating the infogain in eahc case and the most with max infogain is chose the best threshold and feature which is outputted.

5. built_tree():
* is basically splits the current node based on the best split from the best threshold and feature giving new daughter nodes which are then split by calling the function over again. this goes on untill there is only one unique value in a node or the max number of splits allowed is reached.
* we are adding max number of split to prevent the model from overfitting memorizing nodes and asking hyperspecific question
* the return value is a dictionary. hence the the value of each key gets updated each run within the function.

6. predict():
* used to predict the labels of test data.
* it traverses the nested dictionary created by the build tree. it uses the thresholds from the tree on the test data to end up at a final prediction node.
* the isinstance() function traverses in build_tree() function, decision nodes are stored as dictionaries (containing keys for feature, threshold, left, and right trees), while leaf nodes are stored as simple scalar values (the final predicted class). This loop continues to run as long as the current node is a dictionary, meaning a final decision has not yet been reached. when it reachs the scalar values it terminates.
* it takes teh feature and threshold in each iterations.

## Body

In [10]:
df = pd.read_csv("yeast.csv")

X= df.drop(columns=['name']).to_numpy(dtype=float)
y= df['name'].to_numpy()


X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2)
initial_indices = np.arange(X_train.shape[0])

tree = build_tree(X_train, y_train, initial_indices, max_depth=7)
preds = predict(X_test, tree)

accuracy = np.mean(preds == y_test) * 100
print(f"Custom Decision Tree Accuracy: {accuracy:.2f}%\n")

Custom Decision Tree Accuracy: 58.59%

